In [1]:
# Import necessary libraries
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.utils.data.sampler import SubsetRandomSampler
from torchinfo import summary
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
from torchvision.utils import make_grid

from PIL import UnidentifiedImageError

from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split
from sklearn.svm import LinearSVC


In [ ]:
df_dir = '/Users/luryand/Documents/VC/img/task04/PetImages'

cat_files = os.listdir(os.path.join(df_dir, 'Cat'))
dog_files = os.listdir(os.path.join(df_dir, 'Dog'))

In [ ]:
full_dataset = ImageFolder(df_dir, transform=None)
train_size = int(0.7 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_indices, val_indices = torch.utils.data.random_split(
    range(len(full_dataset)), [train_size, val_size],
    generator=torch.Generator().manual_seed(42)

In [ ]:
def safe_loader(path):
    try:
        return Image.open(path).convert('RGB')
    except (UnidentifiedImageError, OSError, ValueError) as e:
        print(f"Erro ao abrir {path}: {e}")
        # Retorna uma imagem preta se der erro
        return Image.fromarray(np.zeros((224, 224, 3), dtype=np.uint8))
    
full_dataset = ImageFolder(df_dir, loader=safe_loader, transform=None)
train_dataset = ImageFolder(df_dir, loader=safe_loader)
val_dataset = ImageFolder(df_dir, loader=safe_loader)

train_sampler = SubsetRandomSampler(train_indices.indices)
val_sampler = SubsetRandomSampler(val_indices.indices)

train_loader = DataLoader(
    train_dataset, batch_size=32, sampler=train_sampler, num_workers=0
)
val_loader = DataLoader(
    val_dataset, batch_size=32, sampler=val_sampler, num_workers=0
)

print(f"Classes: {train_dataset.classes}")
print(f"Total de amostras: {len(full_dataset)}")
print(f"Amostras de treino: {len(train_indices)}")
print(f"Amostras de validação: {len(val_indices)}")